In [2]:
#this dimension belengs to application schema
#creating statecities dimension

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f
import os as os

from dotenv import load_dotenv

In [3]:
env = load_dotenv('/mnt/d/linux/project/vars/.env')

pgjar = os.getenv('POSTGRE_JAR')    
pgconn = os.getenv('POSTGRE_CONN')
pgdriver = os.getenv('PG_DRIVER')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')

In [4]:
#session 

spark = (( SparkSession
            .builder
            .appName('statecities')
            .config('spark.jars', f'{pgjar}')
            .config('spark.driver.extraClassPath', f'{pgjar}')
            .config('spark.executor.extraClassPath', f'{pgjar}')
            .getOrCreate()
        ))

your 131072x1 screen size is bogus. expect trouble
23/10/17 14:40:40 WARN Utils: Your hostname, DOUGPC resolves to a loopback address: 127.0.1.1; using 172.24.129.215 instead (on interface eth0)
23/10/17 14:40:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
23/10/17 14:40:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/10/17 14:40:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
tmp_citiesframe =  ((spark
                        .read
                        .format('jdbc')
                        .option('url', pgconn)
                        .option('driver', pgdriver)
                        .option('user', pguser)
                        .option('password', pgpwd)
                        .option('query', 
                                """ 
                                    select 

                                        apc.cityid,
                                        apc.cityname,
                                        apc.stateprovinceid,
                                        apc.latestrecordedpopulation,
                                        apc._airbyte_ab_id,
                                        apc._airbyte_cities_hashid,
                                        cast(apc._airbyte_emitted_at as date) as _airbyte_emitted_at,
                                        cast(apc._airbyte_normalized_at as date) as _airbyte_normalized_at

                                    from application.cities as apc

                                    left join

                                        application.cities_archive as apca
                                    on apc.cityid = apca.cityid
                                    
                                    """) 
                          )).load()

tmp_citiesframe.show(5, False)

+------+----------+---------------+------------------------+------------------------------------+--------------------------------+-------------------+----------------------+
|cityid|cityname  |stateprovinceid|latestrecordedpopulation|_airbyte_ab_id                      |_airbyte_cities_hashid          |_airbyte_emitted_at|_airbyte_normalized_at|
+------+----------+---------------+------------------------+------------------------------------+--------------------------------+-------------------+----------------------+
|1     |Aaronsburg|39             |613                     |220febd6-5673-4396-80e1-466987cc4e01|facf944602a911fcb66b67dd1ad06120|2023-07-17         |2023-07-17            |
|3     |Abanda    |1              |192                     |ae1e3811-c0a2-4194-9f1f-7aead0097646|8c035a709f265b569747c107f84e9143|2023-07-17         |2023-07-17            |
|4     |Abbeville |42             |5237                    |86f162cb-9599-482e-aa70-936545b59cd9|a7cbd39f728f02e1957904b96e089919|

In [6]:
tmp_stateframe = ((spark
                    .read
                    .format('jdbc')
                    .option('url', pgconn)
                    .option('driver', pgdriver)
                    .option('user', pguser)
                    .option('password', pgpwd)
                    .option('query',
                            """ 
                                select

                                    asp.countryid,
                                    asp.stateprovinceid,
                                    asp.stateprovincename,
                                    asp.salesterritory,
                                    asp.stateprovincecode,
                                    asp.latestrecordedpopulation,
                                    asp._airbyte_ab_id,
                                    asp._airbyte_stateprovinces_hashid,
                                    cast(asp._airbyte_emitted_at as date) as _airbyte_emitted_at,
                                    cast(asp._airbyte_normalized_at as date) as _airbyte_normalized_at
                                from application.stateprovinces as asp
                                
                                left join
                                    application.stateprovinces_archive as aspa
                                on asp.stateprovinceid = aspa.stateprovinceid
                            
                            """) 
                   )).load()

tmp_stateframe.show()

+---------+---------------+-----------------+--------------+-----------------+------------------------+--------------------+------------------------------+-------------------+----------------------+
|countryid|stateprovinceid|stateprovincename|salesterritory|stateprovincecode|latestrecordedpopulation|      _airbyte_ab_id|_airbyte_stateprovinces_hashid|_airbyte_emitted_at|_airbyte_normalized_at|
+---------+---------------+-----------------+--------------+-----------------+------------------------+--------------------+------------------------------+-------------------+----------------------+
|      230|              1|          Alabama|     Southeast|               AL|                 5437278|19e57a2d-0cf8-48e...|          0b01e222063d095da...|         2023-07-17|            2023-07-17|
|      230|              4|         Arkansas|     Southeast|               AR|                 3077747|dbd60e2e-6eda-42b...|          d91c7928c81a5dd96...|         2023-07-17|            2023-07-17|
|    

In [11]:
#joining dataframes based on stateprovinceid column
state_cities = (( 
                 tmp_stateframe
                    .join(tmp_citiesframe,
                          on= 'stateprovinceid',
                          how= 'left')
                    
                    #select and rename columns
                    .select(tmp_stateframe.countryid,
                            tmp_stateframe.stateprovinceid,
                            tmp_stateframe.stateprovincename,
                            tmp_stateframe.salesterritory,
                            tmp_stateframe.stateprovincecode,
                            tmp_citiesframe.cityid,
                            tmp_citiesframe.cityname,
                            tmp_stateframe.latestrecordedpopulation.alias('statepopulation'),
                            
                            #removing null values
                            f.coalesce(tmp_citiesframe.latestrecordedpopulation, f.lit(0)).alias('citypopulation'),
                            tmp_stateframe._airbyte_ab_id,
                            
                            #rename to avoid duplicated results
                            tmp_citiesframe._airbyte_ab_id.alias('city_elt_hashid'),
                            tmp_stateframe._airbyte_stateprovinces_hashid,
                            tmp_citiesframe._airbyte_cities_hashid,
                            tmp_stateframe._airbyte_emitted_at,
                            tmp_stateframe._airbyte_normalized_at)
                 ))


In [8]:
#rename column
statecities_final = (( 
                      state_cities
                        .withColumnRenamed('stateprovinceid', 'stateid')
                        .withColumnRenamed('stateprovincename', 'statename')
                        .withColumnRenamed('_airbyte_ab_id', 'state_elt_hashid')
                        .withColumnRenamed('tmp_citiesframe._airbyte_ab_id', 'city_elt_hashid')
                        .withColumnRenamed('_airbyte_stateprovinces_hashid', 'stateprovince_hashid')
                        .withColumnRenamed('_airbyte_cities_hashid', 'cities_hashid')
                        .withColumnRenamed('_airbyte_emitted_at', 'elt_start')
                        .withColumnRenamed('_airbyte_normalized_at', 'elt_end')
                      ))

statecities_final.show()

+---------+-------+---------+--------------+-----------------+------+--------+---------------+--------------+----------------+---------------+--------------------+-------------+---------+-------+
|countryid|stateid|statename|salesterritory|stateprovincecode|cityid|cityname|statepopulation|citypopulation|state_elt_hashid|city_elt_hashid|stateprovince_hashid|cities_hashid|elt_start|elt_end|
+---------+-------+---------+--------------+-----------------+------+--------+---------------+--------------+----------------+---------------+--------------------+-------------+---------+-------+
+---------+-------+---------+--------------+-----------------+------+--------+---------------+--------------+----------------+---------------+--------------------+-------------+---------+-------+



In [9]:
((
    statecities_final
        .write
            .format('jdbc')
            .option('url', f'{pgconn}')
            .option('driver', f'{pgdriver}')
            .option('user', f'{pguser}')
            .option('password', f'{pgpwd}')
            .option('dbtable', 'application.testeinsert')
        .save(mode= 'Overwrite')
    
))